# O8 — Mechanistic↔behavioral instrument validation (replaces N3)

**N3 failed:** the outcome was constant (Llama W3 accuracy 1/60, Qwen 0/61) — you cannot correlate against a constant.

**This notebook** correlates **per-layer** canonical→W3 gold-token **rank shift** against the **continuous** O5 outcome (`delta_mean_logprob`), not binary correctness. Binary is still reported to show it is degenerate.

### Framing constraint (read before claiming anything)
This is **instrument validation only** ("does the behavioral measure track anything internal"), **never** a mechanism claim about where a split lives. arXiv 2602.04843 (Feb 2026) already did per-layer Mystery-Blocksworld mechanistic analysis.

### Models / families
| | |
|--|--|
| Models | `Qwen/Qwen2.5-1.5B-Instruct`, `Qwen/Qwen2.5-3B-Instruct` |
| Precision | **fp16 ONLY** + `attn_implementation="sdpa"` (no 4-bit/8-bit; see O6) |
| ALGO | frozen adversarial 61 (canonical + W3) |
| GSM | included **iff O7 PASS** (auto from verdict file; override with `INCLUDE_GSM`) |
| BW | **excluded** (gold-token degeneracy) |

### Outputs
- `O8_mech_behavior_link.csv` — per instance × layer
- `O8_layer_profile.csv` — Spearman(rank_shift, y) by layer with cluster-bootstrap CIs
- `O8_framing.txt` — validation-only disclaimer

Clone-family cluster bootstrap (`n_boot=5000`, seed 42), same stack as N3.


In [ ]:
# Colab T4: bitsandbytes for quantized loads. Restart the runtime if
# bitsandbytes was just installed and the kernel has not picked it up.
import sys
import subprocess
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "transformers>=4.44",
        "accelerate>=0.33",
        "bitsandbytes>=0.43",
        "pandas",
        "scipy",
        "networkx",
        "tqdm",
        "huggingface_hub",
    ]
)


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

# ── knobs ────────────────────────────────────────────────────────────────
# Set LIMIT to an int for a smoke test (e.g. 2 items per family). None = full run.
LIMIT = None
DRY_RUN = False          # True: skip GPU, write placeholder rows (pipeline check)
RESUME = True

# Private GitHub clone (Colab secret GITHUB_TOKEN, or env). Public clone works
# without a token. If this notebook is already inside the repo, clone is skipped.
REPO_URL = os.environ.get(
    "RVC_REPO_URL",
    "https://github.com/Adya6714/retrieval-vs-computation.git",
)
REPO_COMMIT = os.environ.get("RVC_REPO_COMMIT", "")  # empty = default branch HEAD

def _secret(name: str) -> str:
    v = os.environ.get(name, "")
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(name) or ""
    except Exception:
        return ""

HF_TOKEN = _secret("HF_TOKEN") or _secret("HUGGING_FACE_HUB_TOKEN")
GH_TOKEN = _secret("GITHUB_TOKEN")

# Llama-3.1-8B-Instruct is gated: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login as _hf_login
        _hf_login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as _hf_exc:
        print("[setup] huggingface login skipped:", _hf_exc)

def _looks_like_repo(p: Path) -> bool:
    return (p / "probes" / "contamination" / "verify.py").is_file() and (
        p / "data" / "problems" / "question_bank_gsm.csv"
    ).is_file()

def _find_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if _looks_like_repo(cand):
            return cand
    colab = Path("/content/retrieval-vs-computation")
    if _looks_like_repo(colab):
        return colab
    return colab

REPO_ROOT = _find_repo()
if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    url = REPO_URL
    if GH_TOKEN and "github.com" in url and url.startswith("https://"):
        url = url.replace("https://", f"https://{GH_TOKEN}@")
    print(f"[setup] cloning {REPO_URL} → {REPO_ROOT}")
    cmd = ["git", "clone", "--depth", "1", url, str(REPO_ROOT)]
    subprocess.check_call(cmd)
    if REPO_COMMIT:
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", REPO_COMMIT])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", REPO_COMMIT])

assert _looks_like_repo(REPO_ROOT), (
    f"Could not find probes/ + question banks under {REPO_ROOT}. "
    "Clone the retrieval-vs-computation repo, or set RVC_REPO_URL / GITHUB_TOKEN."
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUT_DIR = Path("/content/colab_out") if Path("/content").exists() else (REPO_ROOT / "colab_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[setup] REPO_ROOT={REPO_ROOT}")
print(f"[setup] OUT_DIR={OUT_DIR}")
print(f"[setup] LIMIT={LIMIT} DRY_RUN={DRY_RUN} RESUME={RESUME}")


## Knobs, O7 gate, item queue (ALGO 61 + optional GSM)

`INCLUDE_GSM=None` reads O7 verdict (`PASS` → include). Set `True`/`False` to force.


In [ ]:
from __future__ import annotations

import csv
import gc
import re
from typing import Any

import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from probes.common.clones import algo_cluster_map
from probes.common.cluster_inference import cluster_bootstrap_assoc
from probes.contamination.verify import verify_gsm_answer
from probes.contamination.verify_algo import verify_algo

# ── knobs (override setup LIMIT/DRY_RUN/RESUME as needed) ─────────────────
INCLUDE_GSM = None  # None=auto from O7; True/False force
N_BOOT = 5000 if not DRY_RUN else 200
BOOT_SEED = 42
MAX_NEW_TOKENS_GREEDY = 128  # W3 binary correctness check only

MODELS = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
]

PROBE1_TEMPLATE = (
    "Solve the following problem exactly and provide only the final answer "
    "in the required output format. Problem: {problem}. Format instruction: "
    "{family_specific_output_format}."
)
FAMILY_FORMAT = {
    "GSM": (
        "Write the final numerical answer on its own line as #### <number>. "
        "No other text after that tag."
    ),
    "ALGO": (
        "Follow the problem's required output format exactly "
        "(Path: / Count: / Selected: or Total: / Scoops:). No explanation."
    ),
}
FORMAT_KEYWORDS = {
    "path", "count", "selected", "coins", "scoops", "total", "answer",
    "final", "####", "#", ":", "[", "]", "{", "}", ",",
    "path:", "count:", "selected:", "coins:", "scoops:",
}

# Frozen ALGO adversarial pool (rebuild/FROZEN_FILTERS.md) — same as mech notebook.
ALGO_ADV = {
    "CC": [f"CC_{i:02d}" for i in range(1, 11)],
    "SP": [
        "SP_003", "SP_004", "SP_005", "SP_019", "SP_020", "SP_021", "SP_023",
        "SP_024", "SP_026", "SP_027", "SP_028", "SP_029", "SP_030", "SP_037",
        "SP_038", "SP_039", "SP_040", "SP_042", "SP_044", "SP_045", "SP_046",
        "SP_047", "SP_048", "SP_062", "SP_063", "SP_064", "SP_065", "SP_066",
        "SP_068", "SP_069", "SP_070", "SP_071", "SP_072", "SP_073",
    ],
    "WIS": [
        "WIS_003", "WIS_004", "WIS_013", "WIS_014", "WIS_015", "WIS_016",
        "WIS_017", "WIS_018", "WIS_019", "WIS_020", "WIS_023", "WIS_024",
        "WIS_025", "WIS_026", "WIS_027", "WIS_028", "WIS_029",
    ],
}
ALGO_ADV_IDS = ALGO_ADV["CC"] + ALGO_ADV["SP"] + ALGO_ADV["WIS"]
assert len(ALGO_ADV_IDS) == 61

O5_CANDIDATES = [
    OUT_DIR / "O5_teacher_forced_likelihood.csv",
    REPO_ROOT / "results/raw/O5_teacher_forced_likelihood.csv",
    Path("/content/drive/MyDrive/rvc_colab_out/O5_teacher_forced_likelihood.csv"),
]
O7_VERDICT_CANDIDATES = [
    OUT_DIR / "O7_gsm_degeneracy_verdict.txt",
    REPO_ROOT / "results/derived/O7_gsm_degeneracy_verdict.txt",
    OUT_DIR / "O7_gsm_degeneracy_check.csv",
    REPO_ROOT / "results/derived/O7_gsm_degeneracy_check.csv",
    Path("/content/drive/MyDrive/rvc_colab_out/O7_gsm_degeneracy_verdict.txt"),
]

O8_LINK = OUT_DIR / "O8_mech_behavior_link.csv"
O8_PROFILE = OUT_DIR / "O8_layer_profile.csv"
O8_FRAMING = OUT_DIR / "O8_framing.txt"
O8_BINARY = OUT_DIR / "O8_w3_binary_scores.csv"  # per-item greedy W3 correctness

LINK_COLUMNS = [
    "family", "model", "problem_id", "layer", "n_layers",
    "rank_canonical", "rank_w3", "rank_shift_canonical_minus_w3",
    "mean_logprob_canonical", "mean_logprob_w3", "delta_mean_logprob",
    "w3_correct", "binary_degenerate_cell", "clone_family",
    "gold_content_canonical", "gold_content_w3",
    "gold_token_id_canonical", "gold_token_id_w3",
    "gold_token_decoded_canonical", "gold_token_decoded_w3",
    "framing",
]
PROFILE_COLUMNS = [
    "family", "model", "layer", "y",
    "n", "n_clusters",
    "spearman_rho", "ci_low", "ci_high", "p_value",
    "p_value_method", "bootstrap", "n_boot", "seed",
    "y_nunique", "binary_outcome_degenerate", "note", "framing",
]
FRAMING = (
    "Instrument validation only — does the behavioral measure track anything "
    "internal. Not a mechanism claim. See arXiv 2602.04843."
)


def _norm_vt(v: str) -> str:
    v = str(v).strip()
    return "canonical" if v.lower() == "canonical" else v.upper()


def _strip_csv_quotes(text: str) -> str:
    s = str(text)
    if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
        s = s[1:-1]
    return s


def _norm_bank(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str).fillna("")
    df["problem_id"] = df["problem_id"].astype(str).str.strip()
    df["variant_type"] = df["variant_type"].map(_norm_vt)
    df["problem_text"] = df["problem_text"].map(_strip_csv_quotes)
    df["correct_answer"] = df["correct_answer"].map(_strip_csv_quotes)
    return df


def resolve_o7_include_gsm() -> bool:
    if INCLUDE_GSM is not None:
        print(f"[o7] INCLUDE_GSM forced={INCLUDE_GSM}")
        return bool(INCLUDE_GSM)
    for p in O7_VERDICT_CANDIDATES:
        if not p.exists():
            continue
        if p.suffix == ".txt":
            text = p.read_text().upper()
            if "VERDICT: PASS" in text or "\nPASS" in text or text.strip().startswith("PASS"):
                print(f"[o7] PASS from {p}")
                return True
            if "VERDICT: FAIL" in text or "FAIL" in text.split("VERDICT:", 1)[-1][:20]:
                print(f"[o7] FAIL from {p}")
                return False
        if p.suffix == ".csv":
            df = pd.read_csv(p, dtype=str)
            if "verdict" in df.columns and (df["verdict"].str.upper() == "FAIL").any():
                print(f"[o7] FAIL from {p}")
                return False
            if "verdict" in df.columns and (df["verdict"].str.upper() == "PASS").all():
                print(f"[o7] PASS from {p}")
                return True
    # Local preview / O7 not uploaded yet: default PASS (GSM content screen passed in O7 dry-run).
    print("[o7] verdict file not found — defaulting INCLUDE_GSM=True (override if O7 FAIL)")
    return True


def gsm_gold_content(correct_answer: str) -> str:
    s = str(correct_answer).strip()
    s = re.sub(r"^####\s*", "", s).replace(",", "")
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(f)
    except ValueError:
        m = re.findall(r"-?\d+(?:\.\d+)?", s)
        if not m:
            raise ValueError(f"no numeric gold in {correct_answer!r}")
        return m[-1]


def algo_gold_content(problem_id: str, correct_answer: str) -> str:
    s = str(correct_answer)
    pid = str(problem_id).strip().upper()
    if pid.startswith("SP"):
        m = re.search(r"Cost\s*:\s*(-?\d+)", s, flags=re.I)
        if not m:
            raise ValueError(f"{problem_id}: no Cost: gold")
        return m.group(1)
    if pid.startswith("CC"):
        m = re.search(r"(?:Count|Total)\s*:\s*(-?\d+)", s, flags=re.I)
        if not m:
            raise ValueError(f"{problem_id}: no Count:/Total: gold")
        return m.group(1)
    if pid.startswith("WIS"):
        m = re.search(r"Total\s*:\s*(-?\d+)", s, flags=re.I)
        if not m:
            raise ValueError(f"{problem_id}: no Total: gold")
        return m.group(1)
    raise ValueError(f"{problem_id}: unknown ALGO subtype")


def build_user(problem_text: str, family: str) -> str:
    return PROBE1_TEMPLATE.format(
        problem=str(problem_text).strip(),
        family_specific_output_format=FAMILY_FORMAT[family],
    )


def bank_row(df: pd.DataFrame, pid: str, vt: str) -> pd.Series:
    sub = df[(df.problem_id == pid) & (df.variant_type == vt)]
    if sub.empty:
        raise KeyError(f"{pid}/{vt}")
    return sub.iloc[0]


algo_df = _norm_bank(REPO_ROOT / "data/problems/question_bank_algo.csv")
gsm_df = _norm_bank(REPO_ROOT / "data/problems/question_bank_gsm.csv")
paired_algo = sorted(
    set(algo_df.loc[algo_df.variant_type == "canonical", "problem_id"])
    & set(algo_df.loc[algo_df.variant_type == "W3", "problem_id"])
)
ALGO_IDS = [pid for pid in ALGO_ADV_IDS if pid in set(paired_algo)]
assert len(ALGO_IDS) == 61, len(ALGO_IDS)

include_gsm = resolve_o7_include_gsm()
GSM_IDS = sorted(
    set(gsm_df.loc[gsm_df.variant_type == "canonical", "problem_id"])
    & set(gsm_df.loc[gsm_df.variant_type == "W3", "problem_id"])
) if include_gsm else []

cmap = algo_cluster_map()


def clone_for(family: str, pid: str) -> str:
    if family == "ALGO":
        return cmap.get(pid, f"SINGLETON_{pid}")
    return f"SINGLETON_{pid}"


def make_item(family: str, pid: str, vt: str, df: pd.DataFrame) -> dict[str, Any]:
    r = bank_row(df, pid, vt)
    if family == "ALGO":
        gold = algo_gold_content(pid, r["correct_answer"])
    else:
        gold = gsm_gold_content(r["correct_answer"])
    return {
        "family": family,
        "problem_id": pid,
        "variant": vt,
        "problem_text": str(r["problem_text"]),
        "correct_answer": str(r["correct_answer"]),
        "gold_content": gold,
        "problem_subtype": str(r.get("problem_subtype", "")).strip().lower(),
        "difficulty_params": str(r.get("difficulty_params", "{}") or "{}"),
        "clone_family": clone_for(family, pid),
    }


ITEMS: list[dict[str, Any]] = []
for pid in ALGO_IDS:
    for vt in ("canonical", "W3"):
        ITEMS.append(make_item("ALGO", pid, vt, algo_df))
for pid in GSM_IDS:
    for vt in ("canonical", "W3"):
        ITEMS.append(make_item("GSM", pid, vt, gsm_df))

if LIMIT is not None:
    keep_algo = set(ALGO_IDS[:LIMIT])
    keep_gsm = set(GSM_IDS[:LIMIT])
    ITEMS = [
        x for x in ITEMS
        if (x["family"] == "ALGO" and x["problem_id"] in keep_algo)
        or (x["family"] == "GSM" and x["problem_id"] in keep_gsm)
    ]

n_algo = len({x["problem_id"] for x in ITEMS if x["family"] == "ALGO"})
n_gsm = len({x["problem_id"] for x in ITEMS if x["family"] == "GSM"})
print(f"[queue] {len(ITEMS)} prompts  ALGO_ids={n_algo}  GSM_ids={n_gsm}  include_gsm={include_gsm}")
print(pd.DataFrame(ITEMS).groupby(["family", "variant"]).size().unstack(fill_value=0).to_string())
assert not any(x["family"] == "BW" for x in ITEMS), "BW must stay excluded"
O8_FRAMING.write_text(FRAMING + "\n")


## Logit-lens readout + O5 mean_logprob + W3 greedy binary

Per-layer gold-token rank at the last prompt position (same targeting as the mechanistic notebook).  
`delta_mean_logprob = mean_logprob_canonical − mean_logprob_w3` (parallel to `rank_shift_canonical_minus_w3`).  
W3 binary correctness via greedy decode + released verifiers (to demonstrate degeneracy).


In [ ]:
def wrap_chat(tokenizer, user_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        add_generation_prompt=True,
        tokenize=False,
    )


def resolve_target_token(tokenizer, prompt: str, answer: str) -> tuple[int, str, list[int], str]:
    def enc(text: str) -> list[int]:
        return tokenizer.encode(text, add_special_tokens=False)

    prompt_ids = enc(prompt)
    answer_ids_bare = enc(answer)
    candidates: list[tuple[str, int, int, list[int]]] = []
    for sep in ("", " "):
        joint = enc(prompt + sep + answer)
        if len(joint) <= len(prompt_ids):
            continue
        if joint[: len(prompt_ids)] != prompt_ids:
            continue
        rest = joint[len(prompt_ids) :]
        candidates.append((sep, int(rest[0]), len(joint), rest))
    if not candidates:
        if not answer_ids_bare:
            return -1, "", [], "EMPTY"
        tid = int(answer_ids_bare[0])
        return tid, tokenizer.decode([tid]), answer_ids_bare, "FALLBACK"
    candidates.sort(key=lambda c: c[2])
    sep, tid, _, rest = candidates[0]
    return tid, tokenizer.decode([tid]), rest, repr(sep)


def assert_content_gold(decoded: str, family: str) -> None:
    d = decoded.strip().lower()
    compact = d.replace(" ", "")
    if compact in FORMAT_KEYWORDS or d in FORMAT_KEYWORDS:
        raise AssertionError(f"format-keyword gold token: {decoded!r}")
    if family in {"GSM", "ALGO"} and not re.search(r"\d", decoded):
        raise AssertionError(f"{family} gold token must contain a digit: {decoded!r}")


def resolve_continuation(tokenizer, prompt: str, answer: str) -> tuple[list[int], list[int]]:
    def enc(text: str) -> list[int]:
        return tokenizer.encode(text, add_special_tokens=False)

    prompt_ids = enc(prompt)
    for sep in ("", " "):
        joint = enc(prompt + sep + str(answer))
        if len(joint) > len(prompt_ids) and joint[: len(prompt_ids)] == prompt_ids:
            return prompt_ids, joint[len(prompt_ids) :]
    return prompt_ids, enc(str(answer))


@torch.inference_mode()
def readout_layers(model, tokenizer, device, user_text: str, gold_content: str, family: str) -> dict:
    prompt = wrap_chat(tokenizer, user_text)
    tid, decoded, gold_ids, sep_note = resolve_target_token(tokenizer, prompt, gold_content)
    assert_content_gold(decoded, family)
    if DRY_RUN or model is None:
        n = 8 if DRY_RUN else int(getattr(model.config, "num_hidden_layers", 8))
        # Inject item-hash variation so continuous correlations are defined in smoke tests.
        h = abs(hash(gold_content + user_text[:40])) % 97
        ranks = [1 + ((h + i) % 50) for i in range(n)]
        return {
            "gold_token_id": tid if tid >= 0 else 1,
            "gold_token_decoded": decoded or "1",
            "sep_note": "DRY_RUN",
            "n_layers": n,
            "ranks": ranks,
            "mean_logprob": round(-0.2 - 0.01 * (h % 10), 6),
        }

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    out = model(**inputs, output_hidden_states=True, use_cache=False)
    hidden_states = out.hidden_states[1:]  # layer 0 = first block
    W_U = model.lm_head.weight.detach().float()
    ranks = []
    for layer_h in hidden_states:
        h = layer_h[0, -1, :].float()
        logits = h @ W_U.T
        target_logit = logits[tid]
        rank = int((logits > target_logit).sum().item()) + 1
        ranks.append(rank)
    del out, hidden_states, inputs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Teacher-forced mean_logprob of full gold content (O5 primary metric; length-normalized).
    prompt_ids, gold_toks = resolve_continuation(tokenizer, prompt, gold_content)
    n_prompt, n_gold = len(prompt_ids), len(gold_toks)
    if n_gold == 0:
        mean_lp = float("nan")
    else:
        inp = torch.tensor([prompt_ids + gold_toks], dtype=torch.long, device=device)
        out2 = model(input_ids=inp, use_cache=False)
        glo = out2.logits[0, n_prompt - 1 : n_prompt + n_gold - 1].float()
        log_probs = F.log_softmax(glo, dim=-1)
        gt = torch.tensor(gold_toks, device=device, dtype=torch.long)
        sum_lp = float(log_probs.gather(1, gt.unsqueeze(1)).squeeze(1).sum().item())
        mean_lp = sum_lp / n_gold
        del out2, inp
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return {
        "gold_token_id": int(tid),
        "gold_token_decoded": decoded,
        "sep_note": sep_note,
        "n_layers": len(ranks),
        "ranks": ranks,
        "mean_logprob": round(float(mean_lp), 6),
    }


@torch.inference_mode()
def greedy_w3_correct(model, tokenizer, device, item: dict) -> bool:
    """Binary W3 correctness for degeneracy contrast (not the primary outcome)."""
    if item["variant"] != "W3":
        raise ValueError("greedy_w3_correct expects W3 items")
    user = build_user(item["problem_text"], item["family"])
    prompt = wrap_chat(tokenizer, user)
    if DRY_RUN or model is None:
        # Near-constant False → binary degeneracy in smoke (flip ~1/40).
        return (abs(hash(item["problem_id"])) % 40) == 0
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    gen = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS_GREEDY,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = gen[0, inputs["input_ids"].shape[1] :]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    del gen, inputs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if item["family"] == "GSM":
        return bool(verify_gsm_answer(text, item["correct_answer"]))
    ok, _reason, _meta = verify_algo(
        item["problem_id"],
        text,
        item["correct_answer"],
        item["problem_subtype"],
        "W3",
        item["difficulty_params"],
    )
    return bool(ok)


def load_o5_lookup() -> dict[tuple[str, str, str, str], float]:
    """(model, family, problem_id, variant) → mean_logprob from O5 CSV if present."""
    path = next((p for p in O5_CANDIDATES if p.exists()), None)
    if path is None:
        print("[o5] no O5 CSV found — will compute mean_logprob in-session (identical teacher-force)")
        return {}
    df = pd.read_csv(path, dtype=str)
    need = {"model", "family", "problem_id", "variant", "mean_logprob"}
    if not need.issubset(df.columns):
        print(f"[o5] {path} missing columns — ignoring")
        return {}
    out: dict[tuple[str, str, str, str], float] = {}
    for r in df.itertuples(index=False):
        try:
            out[(str(r.model), str(r.family), str(r.problem_id), str(r.variant))] = float(r.mean_logprob)
        except Exception:
            continue
    print(f"[o5] loaded {len(out)} mean_logprob cells from {path}")
    return out


def load_model_fp16(model_id: str):
    assert torch.cuda.is_available() or DRY_RUN, "GPU required unless DRY_RUN"
    tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    if DRY_RUN:
        print(f"[model] DRY_RUN skip weights: {model_id} fp16")
        return tok, None, torch.device("cpu")
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        attn_implementation="sdpa",
        token=HF_TOKEN or True,
    )
    mdl.eval()
    device = next(mdl.parameters()).device
    print(f"[model] {model_id}  fp16 + sdpa  layers={mdl.config.num_hidden_layers}  device={device}")
    return tok, mdl, device


def unload(mdl):
    if mdl is None:
        return
    del mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Run readouts → instance×layer CSV → per-layer Spearman profile

Primary y = `delta_mean_logprob` (continuous, from O5 / in-session).  
Secondary y = `w3_correct` (binary) — expect `binary_outcome_degenerate=True` when y is constant.


In [ ]:
o5_lookup = load_o5_lookup()


def append_link_rows(rows: list[dict[str, Any]]) -> None:
    if not rows:
        return
    write_header = not O8_LINK.exists()
    with O8_LINK.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LINK_COLUMNS, extrasaction="ignore")
        if write_header:
            w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in LINK_COLUMNS})


def done_pairs(path: Path) -> set[tuple[str, str, str]]:
    """Completed (model, family, problem_id) pairs (both variants written)."""
    if not path.exists():
        return set()
    df = pd.read_csv(path, dtype=str)
    if not {"model", "family", "problem_id", "layer"}.issubset(df.columns):
        return set()
    # A pair is done if layer 0 exists (implies full layer stack written together).
    sub = df[pd.to_numeric(df["layer"], errors="coerce") == 0]
    return set(zip(sub["model"], sub["family"], sub["problem_id"]))


done = done_pairs(O8_LINK) if RESUME else set()
print(f"[resume] {len(done)} instance keys in {O8_LINK}")

binary_rows: list[dict[str, Any]] = []

for model_id in MODELS:
    # Unique problem keys still pending for this model
    keys = sorted({(it["family"], it["problem_id"]) for it in ITEMS})
    pending_keys = [(f, p) for f, p in keys if (model_id, f, p) not in done]
    print(f"\n=== {model_id}  pending_instances={len(pending_keys)}/{len(keys)} ===")
    if not pending_keys:
        continue
    tok, mdl, device = load_model_fp16(model_id)
    by_key: dict[tuple[str, str], dict[str, dict]] = {}
    for it in ITEMS:
        by_key.setdefault((it["family"], it["problem_id"]), {})[it["variant"]] = it

    try:
        for fam, pid in tqdm(pending_keys, desc=model_id.split("/")[-1]):
            can_it = by_key[(fam, pid)]["canonical"]
            w3_it = by_key[(fam, pid)]["W3"]
            can_user = build_user(can_it["problem_text"], fam)
            w3_user = build_user(w3_it["problem_text"], fam)
            can_m = readout_layers(mdl, tok, device, can_user, can_it["gold_content"], fam)
            w3_m = readout_layers(mdl, tok, device, w3_user, w3_it["gold_content"], fam)

            # Prefer O5 CSV mean_logprob when present for this model cell.
            mlp_can = o5_lookup.get((model_id, fam, pid, "canonical"), can_m["mean_logprob"])
            mlp_w3 = o5_lookup.get((model_id, fam, pid, "W3"), w3_m["mean_logprob"])
            delta = float(mlp_can) - float(mlp_w3)

            w3_ok = greedy_w3_correct(mdl, tok, device, w3_it)
            binary_rows.append(
                {
                    "family": fam,
                    "model": model_id,
                    "problem_id": pid,
                    "w3_correct": bool(w3_ok),
                    "clone_family": can_it["clone_family"],
                }
            )

            n_layers = min(can_m["n_layers"], w3_m["n_layers"])
            buf = []
            for layer in range(n_layers):
                rc = int(can_m["ranks"][layer])
                rw = int(w3_m["ranks"][layer])
                buf.append(
                    {
                        "family": fam,
                        "model": model_id,
                        "problem_id": pid,
                        "layer": layer,
                        "n_layers": n_layers,
                        "rank_canonical": rc,
                        "rank_w3": rw,
                        "rank_shift_canonical_minus_w3": rc - rw,
                        "mean_logprob_canonical": mlp_can,
                        "mean_logprob_w3": mlp_w3,
                        "delta_mean_logprob": round(delta, 6),
                        "w3_correct": bool(w3_ok),
                        "binary_degenerate_cell": "",  # filled in profile pass
                        "clone_family": can_it["clone_family"],
                        "gold_content_canonical": can_it["gold_content"],
                        "gold_content_w3": w3_it["gold_content"],
                        "gold_token_id_canonical": can_m["gold_token_id"],
                        "gold_token_id_w3": w3_m["gold_token_id"],
                        "gold_token_decoded_canonical": can_m["gold_token_decoded"],
                        "gold_token_decoded_w3": w3_m["gold_token_decoded"],
                        "framing": FRAMING,
                    }
                )
            append_link_rows(buf)
    finally:
        unload(mdl)

if binary_rows:
    pd.DataFrame(binary_rows).drop_duplicates(
        ["family", "model", "problem_id"], keep="last"
    ).to_csv(O8_BINARY, index=False)

link = pd.read_csv(O8_LINK)
print(f"[link] rows={len(link)}  models={sorted(link.model.unique())}")


def profile_block(sub: pd.DataFrame, family: str, model_id: str, layer: int, y_col: str) -> dict:
    x = sub["rank_shift_canonical_minus_w3"]
    y = sub[y_col]
    if y_col == "w3_correct":
        y = y.astype(str).str.lower().isin({"true", "1", "yes"}).astype(int)
    else:
        y = pd.to_numeric(y, errors="coerce")
    x = pd.to_numeric(x, errors="coerce")
    mask = x.notna() & y.notna()
    x, y = x[mask], y[mask]
    clusters = sub.loc[mask, "clone_family"].astype(str).tolist()
    y_nunique = int(pd.Series(y).nunique(dropna=True))
    binary_degen = y_col == "w3_correct" and y_nunique < 2
    note = FRAMING
    if binary_degen:
        note = (
            f"BINARY DEGENERATE: {y_col} is constant "
            f"(nunique={y_nunique}, n={len(y)}, mean={float(y.mean()) if len(y) else float('nan'):.4f}). "
            "Spearman undefined — this is why N3 collapsed; use delta_mean_logprob."
        )
        return {
            "family": family,
            "model": model_id,
            "layer": layer,
            "y": y_col,
            "n": int(len(y)),
            "n_clusters": len(set(clusters)),
            "spearman_rho": "",
            "ci_low": "",
            "ci_high": "",
            "p_value": "",
            "p_value_method": "cluster_bootstrap_two_sided",
            "bootstrap": "cluster_by_clone_family",
            "n_boot": N_BOOT,
            "seed": BOOT_SEED,
            "y_nunique": y_nunique,
            "binary_outcome_degenerate": True,
            "note": note,
            "framing": FRAMING,
        }
    if int(pd.Series(x).nunique(dropna=True)) < 2:
        return {
            "family": family,
            "model": model_id,
            "layer": layer,
            "y": y_col,
            "n": int(len(y)),
            "n_clusters": len(set(clusters)),
            "spearman_rho": "",
            "ci_low": "",
            "ci_high": "",
            "p_value": "",
            "p_value_method": "cluster_bootstrap_two_sided",
            "bootstrap": "cluster_by_clone_family",
            "n_boot": N_BOOT,
            "seed": BOOT_SEED,
            "y_nunique": y_nunique,
            "binary_outcome_degenerate": False,
            "note": "rank_shift constant — correlation undefined; " + FRAMING,
            "framing": FRAMING,
        }
    res = cluster_bootstrap_assoc(
        x, y, clusters, kind="spearman", n_boot=N_BOOT, seed=BOOT_SEED
    )

    def _r(v):
        return round(float(v), 4) if v == v else ""

    return {
        "family": family,
        "model": model_id,
        "layer": layer,
        "y": y_col,
        "n": res["n"],
        "n_clusters": res["n_clusters"],
        "spearman_rho": _r(res["estimate"]),
        "ci_low": _r(res["ci_low"]),
        "ci_high": _r(res["ci_high"]),
        "p_value": _r(res["p_clustered"]),
        "p_value_method": "cluster_bootstrap_two_sided",
        "bootstrap": "cluster_by_clone_family",
        "n_boot": N_BOOT,
        "seed": BOOT_SEED,
        "y_nunique": y_nunique,
        "binary_outcome_degenerate": False,
        "note": note,
        "framing": FRAMING,
    }


profile_rows: list[dict] = []
for (fam, model_id), g in link.groupby(["family", "model"]):
    # Mark binary degeneracy at the instance level for this model×family
    inst = g.drop_duplicates(["problem_id"])
    ybin = inst["w3_correct"].astype(str).str.lower().isin({"true", "1", "yes"})
    bin_degen = int(ybin.nunique()) < 2
    if bin_degen:
        print(
            f"[binary] DEGENERATE  {model_id} / {fam}: "
            f"w3_correct nunique={ybin.nunique()}  "
            f"acc={float(ybin.mean()):.4f}  n={len(ybin)}"
        )
    else:
        print(
            f"[binary] ok variation  {model_id} / {fam}: "
            f"acc={float(ybin.mean()):.4f}  n={len(ybin)}"
        )
    for layer, gl in g.groupby(pd.to_numeric(g["layer"], errors="coerce")):
        if pd.isna(layer):
            continue
        layer_i = int(layer)
        for y_col in ("delta_mean_logprob", "w3_correct"):
            profile_rows.append(profile_block(gl, fam, model_id, layer_i, y_col))

prof = pd.DataFrame(profile_rows, columns=PROFILE_COLUMNS)
prof.to_csv(O8_PROFILE, index=False)

print("\n=== O8_layer_profile.csv (final layer only, preview) ===")
final_layers = (
    link.groupby(["family", "model"])["n_layers"].first().astype(int) - 1
)
preview = []
for (fam, model_id), g in prof.groupby(["family", "model"]):
    fl = int(final_layers.get((fam, model_id), g["layer"].max()))
    preview.append(g[g["layer"] == fl])
if preview:
    print(pd.concat(preview).to_string(index=False))

print(f"\n[wrote] {O8_LINK} ({len(link)} rows)")
print(f"[wrote] {O8_PROFILE} ({len(prof)} rows)")
print(f"[wrote] {O8_FRAMING}")
print("\n" + FRAMING)


## Download / Drive backup

Copy after Colab:
- `O8_mech_behavior_link.csv` → `results/raw/O8_mech_behavior_link.csv`
- `O8_layer_profile.csv` → `results/derived/O8_layer_profile.csv`
- `O8_w3_binary_scores.csv` → `results/raw/O8_w3_binary_scores.csv`
- `O8_framing.txt` → `results/derived/O8_framing.txt`


In [ ]:
_out_files = [O8_LINK, O8_PROFILE, O8_BINARY, O8_FRAMING]
_drive_dir = Path("/content/drive/MyDrive/rvc_colab_out")
if Path("/content").exists() and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("[drive] mount skipped:", exc)
try:
    import shutil as _shutil
    _drive_dir.mkdir(parents=True, exist_ok=True)
    for p in _out_files:
        if p.exists():
            _shutil.copy2(p, _drive_dir / p.name)
            print(f"[backup] {p.name} -> {_drive_dir / p.name}")
except Exception as exc:
    print("[backup] skipped:", exc)
try:
    from google.colab import files as _colab_files  # type: ignore
    for p in _out_files:
        if p.exists():
            _colab_files.download(str(p))
            print(f"[download] {p.name}")
except Exception as exc:
    print("[download] skipped (not Colab or download blocked):", exc)
